In [2]:
pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable
  Using cached lightgbm-4.7.0-py3-none-win_amd64.whl.metadata (18 kB)
Using cached lightgbm-4.7.0-py3-none-win_amd64.whl (1.4 MB)
Note: you may need to restart the kernel to use updated packages.


### M1 model(Sqrt(n)) -> 가중치 적용 및 다음분기 폐업률 예측

In [6]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr


# =========================================================
# 1. 데이터 불러오기
# =========================================================

file_path = (
    r"C:\Users\user\Desktop\AI경진대회\[AI경진대회]DATA\전처리\[새로해보는2]인천_분기x읍면동x업태_다음분기폐업률_전처리_가통합.csv"
)

df = pd.read_csv(file_path)

print("전체 데이터:", df.shape)
print(df["기준분기"].value_counts().sort_index())


# =========================================================
# 2. M1 Feature
#    M0 + 시간순번
# =========================================================

features = [

    # -------------------------
    # 시간
    # -------------------------
    "분기번호",
    "시간순번",              # ★ M1에서 추가

    # -------------------------
    # 업태
    # -------------------------
    "업태구분_2진수",

    # -------------------------
    # 읍면동 × 업태 사업체 구성
    # -------------------------
    "읍면동업태_영업점포수",
    "평균업력개월",
    "업력1년미만_점포비중",
    "평균시설규모",
    "중앙시설규모",
    "소규모점포비중",
    "다중이용업소비중",

    # -------------------------
    # 법정동 수요 / 경쟁 환경
    # -------------------------
    "법정동_실제총인구",
    "법정동_인구증감률_QoQ",
    "법정동_인구증감률_YoY",
    "법정동_인구천명당_전체점포수",
    "법정동_인구천명당_동일업태점포수",

    # -------------------------
    # 읍면동 × 업태 상권 상태
    # -------------------------
    "읍면동업태_분기개업수",
    "읍면동업태_분기폐업수",
    "읍면동업태_점포비중",
    "읍면동업태_직전분기말점포수",
    "읍면동업태_개업률",
    "읍면동업태_폐업률",
    "읍면동업태_점포수증감률",
    "읍면동업태_폐업률_lag1",
    "읍면동업태_폐업률_lag4",
    "읍면동업태_폐업률_4Q평균",
    "읍면동업태_폐업률_전분기대비증감",
]

target = "다음분기_폐업률"

print("\nM1 Feature 수:", len(features))


# =========================================================
# 3. Train / Test
# Train : 2023Q1 ~ 2025Q4
# Test  : 2026Q1 ~ 2026Q2
# =========================================================

train_df = df[df["기준연도"] <= 2025].copy()
test_df = df[df["기준연도"] == 2026].copy()

X_train = train_df[features].copy()
y_train = train_df[target].astype(float)

X_test = test_df[features].copy()
y_test = test_df[target].astype(float)

print("Train:", len(train_df))
print("Test :", len(test_df))


# =========================================================
# 4. Sample Weight
# sqrt(읍면동업태 영업점포수)
# =========================================================

train_weight = np.sqrt(
    train_df["읍면동업태_영업점포수"].astype(float)
)

test_weight = np.sqrt(
    test_df["읍면동업태_영업점포수"].astype(float)
)


# =========================================================
# 5. LightGBM
# M0와 동일한 하이퍼파라미터
# =========================================================

model = lgb.LGBMRegressor(
    objective="regression_l1",

    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=70,

    reg_alpha=0.1,
    reg_lambda=1.0,

    colsample_bytree=0.9,
    subsample=0.9,

    max_depth=-1,
    n_estimators=120,

    random_state=42,
    verbosity=-1,
    force_col_wise=True,
    n_jobs=4
)


# =========================================================
# 6. 학습
# =========================================================

model.fit(
    X_train,
    y_train,
    sample_weight=train_weight
)


# =========================================================
# 7. 예측
# =========================================================

y_pred = model.predict(X_test)

# 폐업률이므로 0~1 범위
y_pred = np.clip(y_pred, 0, 1)


# =========================================================
# 8. 평가
# =========================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

wmae = np.average(
    np.abs(y_test - y_pred),
    weights=test_weight
)

wrmse = np.sqrt(
    np.average(
        (y_test.to_numpy() - y_pred) ** 2,
        weights=test_weight.to_numpy()
    )
)

pearson = pearsonr(
    y_test,
    y_pred
)[0]

spearman = spearmanr(
    y_test,
    y_pred
)[0]

r2 = r2_score(
    y_test,
    y_pred
)


# =========================================================
# 9. 결과 출력
# =========================================================

print("\n==========================================")
print(" M1 : M0 + 시간순번")
print("==========================================")

print(f"MAE      : {mae:.6f} ({mae*100:.3f}%p)")
print(f"RMSE     : {rmse:.6f} ({rmse*100:.3f}%p)")
print(f"WMAE     : {wmae:.6f} ({wmae*100:.3f}%p)")
print(f"WRMSE    : {wrmse:.6f} ({wrmse*100:.3f}%p)")
print(f"Pearson  : {pearson:.3f}")
print(f"Spearman : {spearman:.3f}")
print(f"R2       : {r2:.3f}")


# =========================================================
# 10. Feature Importance
# =========================================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

importance_df["Importance_pct"] = (
    importance_df["Importance"]
    / importance_df["Importance"].sum()
    * 100
)

print("\n===== Feature Importance Top 15 =====")
print(
    importance_df.head(15).to_string(index=False)
)


# 시간순번 중요도 별도 확인
print("\n===== 시간순번 중요도 =====")
print(
    importance_df[
        importance_df["Feature"] == "시간순번"
    ].to_string(index=False)
)


# =========================================================
# 11. 테스트 예측 결과 저장
# =========================================================

result = test_df[
    [
        "기준분기",
        "예측대상분기",
        "군구명_개편후",
        "읍면동_개편후",
        "업태구분명",
        "읍면동업태_영업점포수",
        "다음분기_폐업수",
        "다음분기_폐업률"
    ]
].copy()

result["예측_다음분기폐업률"] = y_pred

result["절대오차"] = np.abs(
    result["다음분기_폐업률"]
    - result["예측_다음분기폐업률"]
)

result.to_csv(
    "M1_시간순번추가_2026Q1_Q2_예측결과.csv",
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# 12. 성능 저장
# =========================================================

metrics_df = pd.DataFrame({
    "Model": ["M1_M0+시간순번"],
    "MAE": [mae],
    "RMSE": [rmse],
    "WMAE": [wmae],
    "WRMSE": [wrmse],
    "Pearson": [pearson],
    "Spearman": [spearman],
    "R2": [r2]
})

metrics_df.to_csv(
    "M1_시간순번추가_성능평가.csv",
    index=False,
    encoding="utf-8-sig"
)

importance_df.to_csv(
    "M1_시간순번추가_FeatureImportance.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료")

전체 데이터: (3642, 44)
기준분기
2023Q1    259
2023Q2    259
2023Q3    259
2023Q4    260
2024Q1    260
2024Q2    260
2024Q3    259
2024Q4    260
2025Q1    261
2025Q2    261
2025Q3    261
2025Q4    261
2026Q1    261
2026Q2    261
Name: count, dtype: int64

M1 Feature 수: 26
Train: 3120
Test : 522

 M1 : M0 + 시간순번
MAE      : 0.013674 (1.367%p)
RMSE     : 0.034240 (3.424%p)
WMAE     : 0.010585 (1.058%p)
WRMSE    : 0.020576 (2.058%p)
Pearson  : 0.142
Spearman : 0.539
R2       : -0.033

===== Feature Importance Top 15 =====
          Feature  Importance  Importance_pct
             분기번호         175       10.416667
   읍면동업태_직전분기말점포수         145        8.630952
           평균업력개월         132        7.857143
      읍면동업태_영업점포수         126        7.500000
   읍면동업태_폐업률_4Q평균         106        6.309524
     읍면동업태_점포수증감률         106        6.309524
        법정동_실제총인구          88        5.238095
      읍면동업태_분기폐업수          73        4.345238
  법정동_인구천명당_전체점포수          69        4.107143
   읍면동업태_폐업률_lag1        

### S20 MODEL (Shrinkage20)

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr


# =========================================================
# 1. 데이터 불러오기
# =========================================================

file_path = (
    r"C:\Users\user\Desktop\AI경진대회\[AI경진대회]DATA\전처리\[새로해보는2]인천_분기x읍면동x업태_다음분기폐업률_전처리_가통합.csv"
)

df = pd.read_csv(file_path)

print("전체 데이터:", df.shape)
print(df["기준분기"].value_counts().sort_index())


# =========================================================
# 2. Feature
#    기존 M1과 동일
# =========================================================

features = [

    # 시간
    "분기번호",
    "시간순번",

    # 업태
    "업태구분_2진수",

    # 읍면동 × 업태 사업체 구성
    "읍면동업태_영업점포수",
    "평균업력개월",
    "업력1년미만_점포비중",
    "평균시설규모",
    "중앙시설규모",
    "소규모점포비중",
    "다중이용업소비중",

    # 법정동 수요 / 경쟁 환경
    "법정동_실제총인구",
    "법정동_인구증감률_QoQ",
    "법정동_인구증감률_YoY",
    "법정동_인구천명당_전체점포수",
    "법정동_인구천명당_동일업태점포수",

    # 읍면동 × 업태 상권 상태
    "읍면동업태_분기개업수",
    "읍면동업태_분기폐업수",
    "읍면동업태_점포비중",
    "읍면동업태_직전분기말점포수",
    "읍면동업태_개업률",
    "읍면동업태_폐업률",
    "읍면동업태_점포수증감률",
    "읍면동업태_폐업률_lag1",
    "읍면동업태_폐업률_lag4",
    "읍면동업태_폐업률_4Q평균",
    "읍면동업태_폐업률_전분기대비증감",
]

raw_target = "다음분기_폐업률"

print("\nFeature 수:", len(features))


# =========================================================
# 3. Train / Test
# Train : 2023Q1 ~ 2025Q4
# Test  : 2026Q1 ~ 2026Q2
# =========================================================

train_df = df[df["기준연도"] <= 2025].copy()
test_df = df[df["기준연도"] == 2026].copy()

print("Train:", len(train_df))
print("Test :", len(test_df))


# =========================================================
# 4. Shrinkage Target 생성
#
# y_shrink = (D + m*p0) / (N + m)
#
# D  : 다음분기 폐업수
# N  : 현재 영업점포수
# p0 : TRAIN 데이터의 업태별 평균 폐업률
# m  : prior strength
#
# 중요:
# p0는 반드시 TRAIN 데이터만 사용
# =========================================================

M = 20

# -----------------------------------------
# 업태별 prior 폐업률 계산
#
# 단순 셀 평균이 아니라
# 총 폐업수 / 총 점포수로 계산
# -----------------------------------------

prior_by_industry = (
    train_df
    .groupby("업태구분_2진수")
    .apply(
        lambda x: x["다음분기_폐업수"].sum()
        / x["읍면동업태_영업점포수"].sum()
    )
    .to_dict()
)

print("\n===== TRAIN 업태별 Prior 폐업률 =====")
for k, v in prior_by_industry.items():
    print(f"업태 {k}: {v:.6f} ({v*100:.3f}%)")


# 각 행에 업태별 prior 부여
train_df["prior_rate"] = (
    train_df["업태구분_2진수"]
    .map(prior_by_industry)
)


# -----------------------------------------
# Shrinkage target 생성
# -----------------------------------------

train_df["다음분기_폐업률_shrink"] = (
    train_df["다음분기_폐업수"]
    + M * train_df["prior_rate"]
) / (
    train_df["읍면동업태_영업점포수"]
    + M
)


# 원래 target과 비교
print("\n===== Shrinkage Target 확인 =====")

check_cols = [
    "업태구분명",
    "읍면동업태_영업점포수",
    "다음분기_폐업수",
    "다음분기_폐업률",
    "prior_rate",
    "다음분기_폐업률_shrink"
]

print(
    train_df[
        check_cols
    ]
    .sort_values(
        "읍면동업태_영업점포수"
    )
    .head(20)
    .to_string(index=False)
)


# =========================================================
# 5. X / y 생성
# =========================================================

X_train = train_df[features].copy()

# ★ 기존 raw 폐업률이 아니라 shrinkage target 학습
y_train = train_df[
    "다음분기_폐업률_shrink"
].astype(float)

X_test = test_df[features].copy()

# 평가는 원래 실제 폐업률로 수행
y_test = test_df[
    raw_target
].astype(float)


# =========================================================
# 6. Sample Weight
#
# 학습:
# sqrt(N)
#
# 평가:
# N 자체를 별도로 사용
# =========================================================

train_weight = np.sqrt(
    train_df[
        "읍면동업태_영업점포수"
    ].astype(float)
)

# 운영적 WMAE / WRMSE 평가용
eval_weight = test_df[
    "읍면동업태_영업점포수"
].astype(float)


# =========================================================
# 7. LightGBM
# 기존 M1과 동일한 하이퍼파라미터
# =========================================================

model = lgb.LGBMRegressor(

    objective="regression_l1",

    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=70,

    reg_alpha=0.1,
    reg_lambda=1.0,

    colsample_bytree=0.9,
    subsample=0.9,

    max_depth=-1,
    n_estimators=120,

    random_state=42,
    verbosity=-1,
    force_col_wise=True,
    n_jobs=4
)


# =========================================================
# 8. 학습
# =========================================================

model.fit(
    X_train,
    y_train,
    sample_weight=train_weight
)


# =========================================================
# 9. 예측
# =========================================================

y_pred = model.predict(X_test)

# 폐업률이므로 0~1
y_pred = np.clip(
    y_pred,
    0,
    1
)


# =========================================================
# 10. 평가
#
# 중요:
# 학습은 shrinkage target으로 했지만
# 최종 평가는 실제 관측 폐업률로 수행
# =========================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

# 실제 점포수 N 가중
wmae = np.average(
    np.abs(
        y_test.to_numpy()
        - y_pred
    ),
    weights=eval_weight.to_numpy()
)

wrmse = np.sqrt(
    np.average(
        (
            y_test.to_numpy()
            - y_pred
        ) ** 2,
        weights=eval_weight.to_numpy()
    )
)

pearson = pearsonr(
    y_test,
    y_pred
)[0]

spearman = spearmanr(
    y_test,
    y_pred
)[0]

r2 = r2_score(
    y_test,
    y_pred
)


# =========================================================
# 11. 결과 출력
# =========================================================

print("\n==========================================")
print(" S20 : Shrinkage Target + sqrt(N)")
print("==========================================")

print(f"Prior strength M = {M}")

print(
    f"MAE      : {mae:.6f} "
    f"({mae*100:.3f}%p)"
)

print(
    f"RMSE     : {rmse:.6f} "
    f"({rmse*100:.3f}%p)"
)

print(
    f"WMAE(N)  : {wmae:.6f} "
    f"({wmae*100:.3f}%p)"
)

print(
    f"WRMSE(N) : {wrmse:.6f} "
    f"({wrmse*100:.3f}%p)"
)

print(f"Pearson  : {pearson:.3f}")
print(f"Spearman : {spearman:.3f}")
print(f"R2       : {r2:.3f}")


# =========================================================
# 12. 점포수 구간별 성능
#
# Shrinkage의 핵심 목적:
# 작은 상권의 예측오차가 실제로 줄었는지 확인
# =========================================================

evaluation_df = test_df.copy()

evaluation_df[
    "예측_다음분기폐업률"
] = y_pred

evaluation_df[
    "절대오차"
] = np.abs(
    evaluation_df["다음분기_폐업률"]
    - evaluation_df["예측_다음분기폐업률"]
)

evaluation_df["점포수구간"] = pd.cut(
    evaluation_df[
        "읍면동업태_영업점포수"
    ],
    bins=[
        0,
        10,
        30,
        100,
        np.inf
    ],
    labels=[
        "N<10",
        "10<=N<30",
        "30<=N<100",
        "N>=100"
    ],
    right=False
)


size_metrics = []

for group, g in evaluation_df.groupby(
    "점포수구간",
    observed=True
):

    group_mae = mean_absolute_error(
        g["다음분기_폐업률"],
        g["예측_다음분기폐업률"]
    )

    group_rmse = np.sqrt(
        mean_squared_error(
            g["다음분기_폐업률"],
            g["예측_다음분기폐업률"]
        )
    )

    size_metrics.append({
        "점포수구간": group,
        "관측치수": len(g),
        "MAE": group_mae,
        "RMSE": group_rmse
    })


size_metrics_df = pd.DataFrame(
    size_metrics
)

print("\n===== 점포수 구간별 성능 =====")
print(
    size_metrics_df.to_string(
        index=False
    )
)


# =========================================================
# 13. Feature Importance
# =========================================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

importance_df[
    "Importance_pct"
] = (
    importance_df["Importance"]
    / importance_df["Importance"].sum()
    * 100
)

print(
    "\n===== Feature Importance Top 15 ====="
)

print(
    importance_df
    .head(15)
    .to_string(index=False)
)


# =========================================================
# 14. 테스트 예측 결과 저장
# =========================================================

result = test_df[
    [
        "기준분기",
        "예측대상분기",
        "군구명_개편후",
        "읍면동_개편후",
        "업태구분명",
        "읍면동업태_영업점포수",
        "다음분기_폐업수",
        "다음분기_폐업률"
    ]
].copy()

result[
    "예측_다음분기폐업률"
] = y_pred

result[
    "절대오차"
] = np.abs(
    result["다음분기_폐업률"]
    - result["예측_다음분기폐업률"]
)

result.to_csv(
    "S20_Shrinkage_2026Q1_Q2_예측결과.csv",
    index=False,
    encoding="utf-8-sig"
)


# =========================================================
# 15. 성능 저장
# =========================================================

metrics_df = pd.DataFrame({
    "Model": [
        "S20_Shrinkage+sqrtN"
    ],
    "M": [M],
    "MAE": [mae],
    "RMSE": [rmse],
    "WMAE_N": [wmae],
    "WRMSE_N": [wrmse],
    "Pearson": [pearson],
    "Spearman": [spearman],
    "R2": [r2]
})

metrics_df.to_csv(
    "S20_Shrinkage_성능평가.csv",
    index=False,
    encoding="utf-8-sig"
)

size_metrics_df.to_csv(
    "S20_Shrinkage_점포수구간별성능.csv",
    index=False,
    encoding="utf-8-sig"
)

importance_df.to_csv(
    "S20_Shrinkage_FeatureImportance.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료")

전체 데이터: (3642, 44)
기준분기
2023Q1    259
2023Q2    259
2023Q3    259
2023Q4    260
2024Q1    260
2024Q2    260
2024Q3    259
2024Q4    260
2025Q1    261
2025Q2    261
2025Q3    261
2025Q4    261
2026Q1    261
2026Q2    261
Name: count, dtype: int64

Feature 수: 26
Train: 3120
Test : 522

===== TRAIN 업태별 Prior 폐업률 =====
업태 0: 0.026059 (2.606%)
업태 1: 0.035179 (3.518%)

===== Shrinkage Target 확인 =====
업태구분명  읍면동업태_영업점포수  다음분기_폐업수  다음분기_폐업률  prior_rate  다음분기_폐업률_shrink
   카페            1         0       0.0    0.035179         0.033504
  음식점            1         0       0.0    0.026059         0.024818
   카페            1         0       0.0    0.035179         0.033504
   카페            1         0       0.0    0.035179         0.033504
   카페            1         0       0.0    0.035179         0.033504
   카페            1         0       0.0    0.035179         0.033504
   카페            1         0       0.0    0.035179         0.033504
   카페            1         0       0.0    0.035179        

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\115693483.py:115: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



 S20 : Shrinkage Target + sqrt(N)
Prior strength M = 20
MAE      : 0.021135 (2.114%p)
RMSE     : 0.034408 (3.441%p)
WMAE(N)  : 0.007726 (0.773%p)
WRMSE(N) : 0.011797 (1.180%p)
Pearson  : 0.033
Spearman : 0.013
R2       : -0.043

===== 점포수 구간별 성능 =====
    점포수구간  관측치수      MAE     RMSE
     N<10   135 0.034567 0.055167
 10<=N<30   114 0.028574 0.035766
30<=N<100   123 0.015212 0.018779
   N>=100   150 0.008249 0.010939

===== Feature Importance Top 15 =====
          Feature  Importance  Importance_pct
      읍면동업태_영업점포수         160        9.523810
             분기번호         125        7.440476
   읍면동업태_직전분기말점포수         125        7.440476
        법정동_실제총인구         106        6.309524
           평균업력개월          99        5.892857
      읍면동업태_분기폐업수          82        4.880952
   읍면동업태_폐업률_4Q평균          81        4.821429
     읍면동업태_점포수증감률          71        4.226190
             시간순번          64        3.809524
       읍면동업태_점포비중          59        3.511905
        읍면동업태_폐업률          56   

### m값 비교 모델

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr


# ============================================================
# 1. 데이터 불러오기
# ============================================================

file_path = (
    r"C:\Users\user\Desktop\AI경진대회\[AI경진대회]DATA\전처리"
    r"\[새로해보는2]인천_분기x읍면동x업태_다음분기폐업률_전처리_가통합.csv"
)

df = pd.read_csv(file_path)

print("전체 데이터:", df.shape)
print("\n분기별 행 수")
print(df["기준분기"].value_counts().sort_index())


# ============================================================
# 2. M1 Feature
# ============================================================

features = [

    # 시간
    "분기번호",
    "시간순번",

    # 업태
    "업태구분_2진수",

    # 읍면동 × 업태 사업체 구성
    "읍면동업태_영업점포수",
    "평균업력개월",
    "업력1년미만_점포비중",
    "평균시설규모",
    "중앙시설규모",
    "소규모점포비중",
    "다중이용업소비중",

    # 법정동 수요 / 경쟁 환경
    "법정동_실제총인구",
    "법정동_인구증감률_QoQ",
    "법정동_인구증감률_YoY",
    "법정동_인구천명당_전체점포수",
    "법정동_인구천명당_동일업태점포수",

    # 읍면동 × 업태 상권 상태
    "읍면동업태_분기개업수",
    "읍면동업태_분기폐업수",
    "읍면동업태_점포비중",
    "읍면동업태_직전분기말점포수",
    "읍면동업태_개업률",
    "읍면동업태_폐업률",
    "읍면동업태_점포수증감률",
    "읍면동업태_폐업률_lag1",
    "읍면동업태_폐업률_lag4",
    "읍면동업태_폐업률_4Q평균",
    "읍면동업태_폐업률_전분기대비증감",
]

print("\nFeature 수:", len(features))


# ============================================================
# 3. Shrinkage 후보
# ============================================================

m_candidates = [0, 5, 10, 20, 30, 50]


# ============================================================
# 4. Expanding-window CV
#
# 2026년은 최종 Test이므로 여기서 절대 사용하지 않음
#
# 최소 2023~2024Q1까지 학습 후
# 2024Q2부터 한 분기씩 검증
# ============================================================

validation_quarters = [
    "2024Q2",
    "2024Q3",
    "2024Q4",
    "2025Q1",
    "2025Q2",
    "2025Q3",
    "2025Q4",
]

print("\nValidation quarters:")
print(validation_quarters)


# ============================================================
# 5. LightGBM 설정
#    기존 M1과 동일
# ============================================================

params = dict(
    objective="regression_l1",
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=70,
    reg_alpha=0.1,
    reg_lambda=1.0,
    colsample_bytree=0.9,
    subsample=0.9,
    max_depth=-1,
    n_estimators=120,
    random_state=42,
    verbosity=-1,
    force_col_wise=True,
    n_jobs=4,
)


# ============================================================
# 6. 분기 순서
# ============================================================

quarter_order = {
    q: i
    for i, q in enumerate(
        sorted(
            df["기준분기"].unique(),
            key=lambda x: (
                int(x[:4]),
                int(x[-1])
            )
        )
    )
}

df["_quarter_order"] = df["기준분기"].map(quarter_order)


# ============================================================
# 7. 안전한 상관계수 함수
# ============================================================

def safe_corr(y_true, y_pred, method="pearson"):

    if len(y_true) < 2:
        return np.nan

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan

    if method == "pearson":
        return pearsonr(y_true, y_pred)[0]

    return spearmanr(y_true, y_pred)[0]


# ============================================================
# 8. CV 시작
# ============================================================

all_results = []
size_results = []


for m in m_candidates:

    print("\n")
    print("=" * 70)
    print(f"Shrinkage m = {m}")
    print("=" * 70)

    for val_q in validation_quarters:

        val_order = quarter_order[val_q]

        # ----------------------------------------------------
        # 해당 validation 분기보다 과거 데이터만 Train
        # ----------------------------------------------------

        train_fold = df[
            (df["_quarter_order"] < val_order)
            & (df["기준연도"] <= 2025)
        ].copy()

        val_fold = df[
            df["기준분기"] == val_q
        ].copy()


        # ----------------------------------------------------
        # 8-1. Prior 계산
        #
        # 반드시 현재 fold의 TRAIN 데이터만 사용
        #
        # p0 = 업태별 총 폐업수 / 총 영업점포수
        # ----------------------------------------------------

        prior_by_industry = (
            train_fold
            .groupby("업태구분_2진수")
            .apply(
                lambda x:
                x["다음분기_폐업수"].sum()
                /
                x["읍면동업태_영업점포수"].sum()
            )
            .to_dict()
        )


        train_fold["prior_rate"] = (
            train_fold["업태구분_2진수"]
            .map(prior_by_industry)
        )


        # ----------------------------------------------------
        # 8-2. Shrinkage Target
        #
        # m = 0이면 원래 폐업률과 정확히 동일
        # ----------------------------------------------------

        if m == 0:

            train_fold["target_shrink"] = (
                train_fold["다음분기_폐업률"]
            )

        else:

            train_fold["target_shrink"] = (
                train_fold["다음분기_폐업수"]
                + m * train_fold["prior_rate"]
            ) / (
                train_fold["읍면동업태_영업점포수"]
                + m
            )


        # ----------------------------------------------------
        # 8-3. X / y
        # ----------------------------------------------------

        X_train = train_fold[features].copy()

        y_train = (
            train_fold["target_shrink"]
            .astype(float)
        )

        X_val = val_fold[features].copy()

        # ★ 검증은 항상 원래 관측 폐업률
        y_val = (
            val_fold["다음분기_폐업률"]
            .astype(float)
        )


        # ----------------------------------------------------
        # 8-4. 학습 Weight = sqrt(N)
        # ----------------------------------------------------

        train_weight = np.sqrt(
            train_fold[
                "읍면동업태_영업점포수"
            ].astype(float)
        )


        # ----------------------------------------------------
        # 8-5. 모델 학습
        # ----------------------------------------------------

        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_train,
            y_train,
            sample_weight=train_weight
        )


        # ----------------------------------------------------
        # 8-6. Validation 예측
        # ----------------------------------------------------

        pred = model.predict(X_val)

        pred = np.clip(
            pred,
            0,
            1
        )


        # ----------------------------------------------------
        # 8-7. 전체 성능
        # ----------------------------------------------------

        y_true = y_val.to_numpy()

        N_val = (
            val_fold[
                "읍면동업태_영업점포수"
            ]
            .astype(float)
            .to_numpy()
        )


        mae = mean_absolute_error(
            y_true,
            pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_true,
                pred
            )
        )


        # 실제 점포수 N 기준 평가
        wmae = np.average(
            np.abs(y_true - pred),
            weights=N_val
        )

        wrmse = np.sqrt(
            np.average(
                (y_true - pred) ** 2,
                weights=N_val
            )
        )


        pearson = safe_corr(
            y_true,
            pred,
            "pearson"
        )

        spearman = safe_corr(
            y_true,
            pred,
            "spearman"
        )

        r2 = r2_score(
            y_true,
            pred
        )


        all_results.append({

            "m": m,
            "Validation분기": val_q,

            "Train행수": len(train_fold),
            "Validation행수": len(val_fold),

            "MAE": mae,
            "RMSE": rmse,

            "WMAE_N": wmae,
            "WRMSE_N": wrmse,

            "Pearson": pearson,
            "Spearman": spearman,
            "R2": r2
        })


        # ----------------------------------------------------
        # 8-8. 점포수 구간별 성능
        # ----------------------------------------------------

        temp = val_fold[
            [
                "읍면동업태_영업점포수",
                "다음분기_폐업률"
            ]
        ].copy()

        temp["prediction"] = pred


        temp["점포수구간"] = pd.cut(

            temp["읍면동업태_영업점포수"],

            bins=[
                0,
                10,
                30,
                100,
                np.inf
            ],

            labels=[
                "N<10",
                "10<=N<30",
                "30<=N<100",
                "N>=100"
            ],

            right=False
        )


        for size_group, g in temp.groupby(
            "점포수구간",
            observed=True
        ):

            group_mae = mean_absolute_error(
                g["다음분기_폐업률"],
                g["prediction"]
            )

            group_rmse = np.sqrt(
                mean_squared_error(
                    g["다음분기_폐업률"],
                    g["prediction"]
                )
            )


            size_results.append({

                "m": m,

                "Validation분기": val_q,

                "점포수구간": size_group,

                "관측치수": len(g),

                "MAE": group_mae,

                "RMSE": group_rmse
            })


        print(
            f"{val_q} | "
            f"MAE={mae*100:.3f}%p | "
            f"WMAE={wmae*100:.3f}%p | "
            f"Spearman={spearman:.3f} | "
            f"R2={r2:.3f}"
        )


# ============================================================
# 9. Fold별 결과 DataFrame
# ============================================================

cv_results = pd.DataFrame(
    all_results
)

size_results_df = pd.DataFrame(
    size_results
)


# ============================================================
# 10. m별 평균 성능
# ============================================================

summary = (
    cv_results
    .groupby("m")
    .agg(
        Fold수=("Validation분기", "count"),

        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),

        평균_WMAE_N=("WMAE_N", "mean"),
        평균_WRMSE_N=("WRMSE_N", "mean"),

        평균_Pearson=("Pearson", "mean"),
        평균_Spearman=("Spearman", "mean"),

        평균_R2=("R2", "mean"),

        MAE_표준편차=("MAE", "std"),
        WMAE_표준편차=("WMAE_N", "std"),
    )
    .reset_index()
)


# ============================================================
# 11. 보기 편하게 %p 열 추가
# ============================================================

summary["평균_MAE_%p"] = (
    summary["평균_MAE"] * 100
)

summary["평균_RMSE_%p"] = (
    summary["평균_RMSE"] * 100
)

summary["평균_WMAE_%p"] = (
    summary["평균_WMAE_N"] * 100
)

summary["평균_WRMSE_%p"] = (
    summary["평균_WRMSE_N"] * 100
)


# ============================================================
# 12. 점포수 구간별 m 성능
# ============================================================

size_summary = (
    size_results_df
    .groupby(
        [
            "m",
            "점포수구간"
        ],
        observed=True
    )
    .agg(
        평균_MAE=("MAE", "mean"),
        평균_RMSE=("RMSE", "mean"),
        평균_관측치수=("관측치수", "mean")
    )
    .reset_index()
)


size_summary["평균_MAE_%p"] = (
    size_summary["평균_MAE"] * 100
)

size_summary["평균_RMSE_%p"] = (
    size_summary["평균_RMSE"] * 100
)


# ============================================================
# 13. 결과 출력
# ============================================================

print("\n")
print("=" * 90)
print("m별 Expanding-window CV 평균 성능")
print("=" * 90)

print(
    summary[
        [
            "m",
            "평균_MAE_%p",
            "평균_RMSE_%p",
            "평균_WMAE_%p",
            "평균_WRMSE_%p",
            "평균_Pearson",
            "평균_Spearman",
            "평균_R2"
        ]
    ]
    .sort_values("평균_WMAE_%p")
    .to_string(index=False)
)


print("\n")
print("=" * 90)
print("m별 × 점포수 구간별 성능")
print("=" * 90)

print(
    size_summary[
        [
            "m",
            "점포수구간",
            "평균_MAE_%p",
            "평균_RMSE_%p",
            "평균_관측치수"
        ]
    ]
    .to_string(index=False)
)


# ============================================================
# 14. 기준 M=0 대비 개선율
# ============================================================

baseline = (
    summary[
        summary["m"] == 0
    ]
    .iloc[0]
)


summary["MAE_개선율_%"] = (
    (
        baseline["평균_MAE"]
        - summary["평균_MAE"]
    )
    /
    baseline["평균_MAE"]
    * 100
)


summary["WMAE_개선율_%"] = (
    (
        baseline["평균_WMAE_N"]
        - summary["평균_WMAE_N"]
    )
    /
    baseline["평균_WMAE_N"]
    * 100
)


summary["RMSE_개선율_%"] = (
    (
        baseline["평균_RMSE"]
        - summary["평균_RMSE"]
    )
    /
    baseline["평균_RMSE"]
    * 100
)


print("\n")
print("=" * 90)
print("M=0 대비 개선율")
print("=" * 90)

print(
    summary[
        [
            "m",
            "MAE_개선율_%",
            "RMSE_개선율_%",
            "WMAE_개선율_%"
        ]
    ]
    .sort_values("m")
    .to_string(index=False)
)


# ============================================================
# 15. 결과 저장
# ============================================================

cv_results.to_csv(
    "Shrinkage_m비교_CV_Fold별성능.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    "Shrinkage_m비교_CV_평균성능.csv",
    index=False,
    encoding="utf-8-sig"
)

size_results_df.to_csv(
    "Shrinkage_m비교_점포수구간별_Fold성능.csv",
    index=False,
    encoding="utf-8-sig"
)

size_summary.to_csv(
    "Shrinkage_m비교_점포수구간별_평균성능.csv",
    index=False,
    encoding="utf-8-sig"
)


print("\n저장 완료")

전체 데이터: (3642, 44)

분기별 행 수
기준분기
2023Q1    259
2023Q2    259
2023Q3    259
2023Q4    260
2024Q1    260
2024Q2    260
2024Q3    259
2024Q4    260
2025Q1    261
2025Q2    261
2025Q3    261
2025Q4    261
2026Q1    261
2026Q2    261
Name: count, dtype: int64

Feature 수: 26

Validation quarters:
['2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']


Shrinkage m = 0
2024Q2 | MAE=1.835%p | WMAE=0.751%p | Spearman=0.612 | R2=-0.031


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\1418057887.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\1418057887.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


2024Q3 | MAE=2.974%p | WMAE=1.874%p | Spearman=0.469 | R2=-0.209
2024Q4 | MAE=2.003%p | WMAE=1.378%p | Spearman=0.459 | R2=-0.077


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\1418057887.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\1418057887.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


2025Q1 | MAE=1.783%p | WMAE=0.928%p | Spearman=0.491 | R2=-0.062
2025Q2 | MAE=1.399%p | WMAE=0.774%p | Spearman=0.499 | R2=-0.028


...

2025Q1 | MAE=2.584%p | WMAE=0.964%p | Spearman=-0.018 | R2=-0.020
2025Q2 | MAE=2.264%p | WMAE=0.818%p | Spearman=-0.065 | R2=-0.106


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\1418057887.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\1418057887.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


2025Q3 | MAE=3.328%p | WMAE=1.099%p | Spearman=0.133 | R2=0.002
2025Q4 | MAE=2.521%p | WMAE=1.021%p | Spearman=0.080 | R2=-0.023


m별 Expanding-window CV 평균 성능
 m  평균_MAE_%p  평균_RMSE_%p  평균_WMAE_%p  평균_WRMSE_%p  평균_Pearson  평균_Spearman     평균_R2
 0   2.049207    5.250083    1.097822     1.652994    0.119401     0.508869 -0.067248
50   2.780100    5.124114    1.102248     1.605859    0.118520     0.056851 -0.031892
 5   2.471803    5.123537    1.116320     1.640769    0.123703     0.321925 -0.019373
10   2.596293    5.108671    1.119243     1.627219    0.123221     0.236077 -0.015642
30   2.745454    5.115920    1.128910     1.626883    0.120292     0.100189 -0.024782
20   2.697059    5.113423    1.135228     1.634824    0.121558     0.157236 -0.020986


m별 × 점포수 구간별 성능
 m     점포수구간  평균_MAE_%p  평균_RMSE_%p   평균_관측치수
 0  10<=N<30   2.697409    4.184934 58.000000
 0 30<=N<100   1.950672    2.536747 61.714286
 0      N<10   2.624235    8.975670 64.857143
 0    N>=100   1.141287    1.494262 

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26320\1418057887.py:210: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



저장 완료


### M1 model -> 가중치 적용 + 다음 분기 폐업수 예측
시간순번 변수 제외

In [5]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr


# =========================================================
# 1. 데이터 불러오기
# =========================================================

file_path = (
    r"C:\Users\user\Desktop\AI경진대회\[AI경진대회]DATA\전처리\[새로해보는2]인천_분기x읍면동x업태_다음분기폐업률_전처리_가통합.csv"
)

df = pd.read_csv(file_path)

print("전체 데이터:", df.shape)
print(df["기준분기"].value_counts().sort_index())


# =========================================================
# 2. 피처 및 타겟 설정
# =========================================================

features = [
    # 시간 (시간순번 제거)
    "분기번호",
    
    # 업태
    "업태구분_2진수",
    
    # 읍면동 × 업태 사업체 구성
    "읍면동업태_영업점포수",
    "평균업력개월",
    "업력1년미만_점포비중",
    "평균시설규모",
    "중앙시설규모",
    "소규모점포비중",
    "다중이용업소비중",
    
    # 법정동 수요 / 경쟁 환경
    "법정동_실제총인구",
    "법정동_인구증감률_QoQ",
    "법정동_인구증감률_YoY",
    "법정동_인구천명당_전체점포수",
    "법정동_인구천명당_동일업태점포수",
    
    # 읍면동 × 업태 상권 상태
    "읍면동업태_분기개업수",
    "읍면동업태_분기폐업수",
    "읍면동업태_점포비중",
    "읍면동업태_직전분기말점포수",
    "읍면동업태_개업률",
    "읍면동업태_폐업률",
    "읍면동업태_점포수증감률",
    "읍면동업태_폐업률_lag1",
    "읍면동업태_폐업률_lag4",
    "읍면동업태_폐업률_4Q평균",
    "읍면동업태_폐업률_전분기대비증감",
]

# 타겟을 폐업수(건수)로 변경
target_count = "다음분기_폐업수"
# 최종 평가는 여전히 폐업률로 진행하기 위해 별도 저장
target_rate_eval = "다음분기_폐업률" 

print("\nFeature 수:", len(features))


# =========================================================
# 3. Train / Test 분할
# =========================================================

train_df = df[df["기준연도"] <= 2025].copy()
test_df = df[df["기준연도"] == 2026].copy()

X_train = train_df[features].copy()
y_train_count = train_df[target_count].astype(float)

X_test = test_df[features].copy()
y_test_count = test_df[target_count].astype(float)

# 평가를 위한 실제 폐업률
y_test_rate = test_df[target_rate_eval].astype(float) 

print("Train:", len(train_df))
print("Test :", len(test_df))


# =========================================================
# 4. LightGBM (Poisson 모델)
# =========================================================

model = lgb.LGBMRegressor(
    objective="poisson",       # 건수 예측에 최적화된 포아송 회귀 적용
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=30,      # 과소적합 방지를 위해 30으로 완화
    reg_alpha=0.1,
    reg_lambda=1.0,
    colsample_bytree=0.9,
    subsample=0.9,
    max_depth=-1,
    n_estimators=120,
    random_state=42,
    verbosity=-1,
    force_col_wise=True,
    n_jobs=4
)


# =========================================================
# 5. 학습
# 포아송 회귀는 값의 크기 자체가 비중 역할을 하므로 Weight 생략 가능
# =========================================================

model.fit(X_train, y_train_count)


# =========================================================
# 6. 예측 및 비율 변환
# =========================================================

# 1) 먼저 다음분기 '폐업수'를 예측
pred_count = model.predict(X_test)

# 2) 예측된 폐업수를 기준 영업점포수로 나누어 '폐업률'로 변환
# 분모가 0인 경우를 방지하기 위해 매우 작은 값(1e-9)을 더해주는 것이 안전할 수 있습니다.
pred_rate = pred_count / (X_test["읍면동업태_영업점포수"] + 1e-9)

# 3) 폐업률은 0~1 사이이므로 클리핑 적용
y_pred_rate = np.clip(pred_rate, 0, 1)


# =========================================================
# 7. 평가 (실제 폐업률 vs 예측 폐업률)
# =========================================================

test_weight = np.sqrt(test_df["읍면동업태_영업점포수"].astype(float))

mae = mean_absolute_error(y_test_rate, y_pred_rate)
rmse = np.sqrt(mean_squared_error(y_test_rate, y_pred_rate))

wmae = np.average(np.abs(y_test_rate - y_pred_rate), weights=test_weight)
wrmse = np.sqrt(np.average((y_test_rate - y_pred_rate) ** 2, weights=test_weight))

pearson = pearsonr(y_test_rate, y_pred_rate)[0]
spearman = spearmanr(y_test_rate, y_pred_rate)[0]
r2 = r2_score(y_test_rate, y_pred_rate)

print("\n==========================================")
print(" 타겟변경: 다음분기_폐업수 예측 후 비율 변환 (시간순번 제외)")
print("==========================================")
print(f"MAE      : {mae:.6f} ({mae*100:.3f}%p)")
print(f"RMSE     : {rmse:.6f} ({rmse*100:.3f}%p)")
print(f"WMAE     : {wmae:.6f} ({wmae*100:.3f}%p)")
print(f"WRMSE    : {wrmse:.6f} ({wrmse*100:.3f}%p)")
print(f"Pearson  : {pearson:.3f}")
print(f"Spearman : {spearman:.3f}")
print(f"R2       : {r2:.3f}")


# =========================================================
# 8. Feature Importance
# =========================================================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values("Importance", ascending=False).reset_index(drop=True)
importance_df["Importance_pct"] = (importance_df["Importance"] / importance_df["Importance"].sum() * 100)

print("\n===== Feature Importance Top 15 =====")
print(importance_df.head(15).to_string(index=False))


# =========================================================
# 9. 테스트 예측 결과 저장
# =========================================================

result = test_df[
    [
        "기준분기",
        "예측대상분기",
        "군구명_개편후",
        "읍면동_개편후",
        "업태구분명",
        "읍면동업태_영업점포수",
        "다음분기_폐업수",
        "다음분기_폐업률"
    ]
].copy()

# ★ 추가: 모델이 직접 예측한 '폐업수' 매핑
result["예측_다음분기폐업수"] = pred_count

# 최종 예측률 매핑
result["예측_다음분기폐업률"] = y_pred_rate
result["절대오차"] = np.abs(result["다음분기_폐업률"] - result["예측_다음분기폐업률"])

result.to_csv(
    "Count예측_시간순번제외_비율변환_2026Q1_Q2_예측결과.csv",
    index=False,
    encoding="utf-8-sig"
)
print("\n저장 완료")

전체 데이터: (3642, 44)
기준분기
2023Q1    259
2023Q2    259
2023Q3    259
2023Q4    260
2024Q1    260
2024Q2    260
2024Q3    259
2024Q4    260
2025Q1    261
2025Q2    261
2025Q3    261
2025Q4    261
2026Q1    261
2026Q2    261
Name: count, dtype: int64

Feature 수: 25
Train: 3120
Test : 522

 타겟변경: 다음분기_폐업수 예측 후 비율 변환 (시간순번 제외)
MAE      : 0.041833 (4.183%p)
RMSE     : 0.073504 (7.350%p)
WMAE     : 0.016968 (1.697%p)
WRMSE    : 0.032115 (3.212%p)
Pearson  : -0.098
Spearman : -0.375
R2       : -3.761

===== Feature Importance Top 15 =====
          Feature  Importance  Importance_pct
             분기번호         260       15.476190
      읍면동업태_영업점포수         259       15.416667
   읍면동업태_직전분기말점포수         257       15.297619
        법정동_실제총인구         180       10.714286
           평균업력개월         114        6.785714
   읍면동업태_폐업률_4Q평균          77        4.583333
      읍면동업태_분기개업수          65        3.869048
   읍면동업태_폐업률_lag1          54        3.214286
     읍면동업태_점포수증감률          38        2.261905
법정동_인

### 폐업률 성능평가(이건 안봐도됨)

In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr

# =========================================================
# 성능 평가 지표 계산
# (y_test_rate, y_pred_rate, test_weight 변수가 메모리에 있다고 가정)
# =========================================================

mae = mean_absolute_error(y_test_rate, y_pred_rate)
rmse = np.sqrt(mean_squared_error(y_test_rate, y_pred_rate))

wmae = np.average(np.abs(y_test_rate - y_pred_rate), weights=test_weight)
wrmse = np.sqrt(np.average((y_test_rate - y_pred_rate) ** 2, weights=test_weight))

pearson = pearsonr(y_test_rate, y_pred_rate)[0]
spearman = spearmanr(y_test_rate, y_pred_rate)[0]
r2 = r2_score(y_test_rate, y_pred_rate)

# =========================================================
# 평가 결과 화면 출력
# =========================================================

print("\n==========================================")
print(" 성능 평가 결과 (건수 예측 후 비율 변환, 시간순번 제외)")
print("==========================================")
print(f"MAE      : {mae:.6f} ({mae*100:.3f}%p)")
print(f"RMSE     : {rmse:.6f} ({rmse*100:.3f}%p)")
print(f"WMAE     : {wmae:.6f} ({wmae*100:.3f}%p)")
print(f"WRMSE    : {wrmse:.6f} ({wrmse*100:.3f}%p)")
print(f"Pearson  : {pearson:.3f}")
print(f"Spearman : {spearman:.3f}")
print(f"R2       : {r2:.3f}")

# =========================================================
# 평가 결과 CSV 파일 저장
# =========================================================

metrics_df = pd.DataFrame({
    "Model": ["Count예측_시간순번제외"],
    "MAE": [mae],
    "RMSE": [rmse],
    "WMAE": [wmae],
    "WRMSE": [wrmse],
    "Pearson": [pearson],
    "Spearman": [spearman],
    "R2": [r2]
})

metrics_df.to_csv(
    "Count예측_시간순번제외_성능평가.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n성능 평가 지표 저장 완료: Count예측_시간순번제외_성능평가.csv")


 성능 평가 결과 (건수 예측 후 비율 변환, 시간순번 제외)
MAE      : 0.041833 (4.183%p)
RMSE     : 0.073504 (7.350%p)
WMAE     : 0.016968 (1.697%p)
WRMSE    : 0.032115 (3.212%p)
Pearson  : -0.098
Spearman : -0.375
R2       : -3.761

성능 평가 지표 저장 완료: Count예측_시간순번제외_성능평가.csv


### 폐업수로 성능평가

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from scipy.stats import pearsonr, spearmanr

# =========================================================
# 건수(Count) 기준 가중치 및 성능 평가 지표 계산
# (test_df, y_test_count, pred_count 변수가 메모리에 있다고 가정)
# =========================================================

actual_count = y_test_count
predicted_count = pred_count

# 가중치 다시 정의
test_weight = np.sqrt(test_df["읍면동업태_영업점포수"].astype(float))

# 기본 지표
mae_count = mean_absolute_error(actual_count, predicted_count)
rmse_count = np.sqrt(mean_squared_error(actual_count, predicted_count))

# 가중치 적용 지표 (WMAE, WRMSE)
wmae_count = np.average(np.abs(actual_count - predicted_count), weights=test_weight)
wrmse_count = np.sqrt(np.average((actual_count - predicted_count) ** 2, weights=test_weight))

# 상관계수 및 결정계수
pearson_count = pearsonr(actual_count, predicted_count)[0]
spearman_count = spearmanr(actual_count, predicted_count)[0]
r2_count = r2_score(actual_count, predicted_count)

# =========================================================
# 평가 결과 화면 출력
# =========================================================

print("\n==========================================")
print(" 성능 평가 결과 (건수 예측 모델 - 폐업수 및 가중치 적용)")
print("==========================================")
print(f"MAE      : {mae_count:.6f} 건")
print(f"RMSE     : {rmse_count:.6f} 건")
print(f"WMAE     : {wmae_count:.6f} 건")
print(f"WRMSE    : {wrmse_count:.6f} 건")
print(f"Pearson  : {pearson_count:.3f}")
print(f"Spearman : {spearman_count:.3f}")
print(f"R2       : {r2_count:.3f}")

# =========================================================
# 평가 결과 CSV 파일 저장
# =========================================================

metrics_count_df = pd.DataFrame({
    "Model": ["Count예측_시간순번제외_WMAE포함"],
    "MAE": [mae_count],
    "RMSE": [rmse_count],
    "WMAE": [wmae_count],
    "WRMSE": [wrmse_count],
    "Pearson": [pearson_count],
    "Spearman": [spearman_count],
    "R2": [r2_count]
})

metrics_count_df.to_csv(
    "Count예측_시간순번제외_건수평가_결과.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n성능 평가 지표 저장 완료: Count예측_시간순번제외_건수평가_결과.csv")


 성능 평가 결과 (건수 예측 모델 - 폐업수 및 가중치 적용)
MAE      : 1.237454 건
RMSE     : 2.354096 건
WMAE     : 2.508003 건
WRMSE    : 4.320856 건
Pearson  : 0.957
Spearman : 0.810
R2       : 0.912

성능 평가 지표 저장 완료: Count예측_시간순번제외_건수평가_결과.csv
